In [1]:
import torch
import torch.nn as nn
import esm
import csv
from datetime import datetime
import os
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
timestamp = datetime.now().strftime('%d-%b_%H-%M-%S')

In [2]:
class ESM2_TransformerHead(nn.Module):
    def __init__(self, num_layers=2, nhead=8, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.esm, self.alphabet = esm.pretrained.esm2_t33_650M_UR50D()
        self.batch_converter = self.alphabet.get_batch_converter()
        self.esm.eval()

        for param in self.esm.parameters():
            param.requires_grad = False
        for param in self.esm.layers[-5:].parameters():
            param.requires_grad = True

        encoder_layer = TransformerEncoderLayer(d_model=self.esm.embed_dim, nhead=nhead, dim_feedforward=hidden_dim, dropout=dropout)
        self.transformer = TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(self.esm.embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, seqs): 
        batch_labels = [("seq", s) for s in seqs]
        _, _, tokens = self.batch_converter(batch_labels)
        tokens = tokens.to(next(self.parameters()).device) 

        with torch.no_grad():
            rep = self.esm(tokens, repr_layers=[12])["representations"][12]
        # Don't strip BOS/EOS — keep token 0 (CLS)
        rep = rep.permute(1, 0, 2)       # (L, B, D)
        encoded = self.transformer(rep) # pass through custom transformer
        pooled = encoded[0]             # CLS token is always at position 0


        return self.classifier(pooled)


In [3]:
class MutationDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

    def __len__(self):
        return len(self.sequences)


In [4]:
df = pd.read_csv("datasets/mutation_30mer_dataset153.csv")
patho_df = df[df["Label"] == 1]
benign_df = df[df["Label"] == 0]

min_count = min(len(patho_df), len(benign_df))

pathogenic_balanced = resample(patho_df, replace=False, n_samples=min_count, random_state=13)
benign_balanced = resample(benign_df, replace=False, n_samples=min_count, random_state=13)

balanced_df = pd.concat([pathogenic_balanced, benign_balanced]).sample(frac=1, random_state=13) 

In [5]:
def collate_fn_stringbatch(batch):
    sequences, labels = zip(*batch)
    return list(sequences), torch.tensor(labels)


In [6]:
X = balanced_df["Mut_30mer"].values
y = balanced_df["Label"].values

xtrain, xtest, ytrain, ytest= train_test_split(X, y, test_size=0.15, stratify=y)

training_data = MutationDataset(xtrain, ytrain)
test_data = MutationDataset(xtest, ytest)
 
train_loader = DataLoader(training_data, batch_size=64, shuffle=True, collate_fn=collate_fn_stringbatch)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=collate_fn_stringbatch)


In [7]:
model = ESM2_TransformerHead(num_layers=2).to(device)

c:\Users\brian\Documents\! Grad School\Classes\CSE 566\Mutation-Prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [8]:
model

ESM2_TransformerHead(
  (esm): ESM2(
    (embed_tokens): Embedding(33, 1280, padding_idx=1)
    (layers): ModuleList(
      (0-32): 33 x TransformerLayer(
        (self_attn): MultiheadAttention(
          (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          (rot_emb): RotaryEmbedding()
        )
        (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1280, out_features=5120, bias=True)
        (fc2): Linear(in_features=5120, out_features=1280, bias=True)
        (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      )
    )
    (contact_head): ContactPredictionHead(
      (regression): Linear(in_features=660, out_features=1, bias=True)
      (activati

In [12]:
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()
csv_log_path = f"/custom_esm/metrics_cesm_{timestamp}.csv"
metrics_log = []
train_losses, val_losses = [], []
epochs = 45
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,       # halve the LR
    patience=3,       # wait 3 epochs before reducing
    verbose=True,
    min_lr=1e-6       # don't go lower than this
)

for ep in range(1, epochs + 1):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    print(f"Epoch {ep}: \tTrain Loss = {train_loss:.4f}, Train Accuracy = {train_acc:.4f}")
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    val_loss = test_loss / len(test_loader)
    val_losses.append(val_loss)
    metrics_log.append({
    "epoch": ep,
    "train_loss": train_loss,
    "train_acc": train_acc,
    "val_loss": val_loss,
    "val_acc": val_acc
    })
    scheduler.step(val_loss)
    print(f"\t \t Test Loss = {val_loss:.4f},  Test Accuracy = {val_acc:.4f}\n")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        # Optional: save best model here
        # torch.save(model.state_dict(), "best_model.pt")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"⏹️ Early stopping triggered at epoch {ep}. No improvement for {patience} epochs.")
        break





c:\Users\brian\Documents\! Grad School\Classes\CSE 566\Mutation-Prediction\venv\Lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 1: 	Train Loss = 0.6560, Train Accuracy = 0.6120
	 	 Test Loss = 0.6395,  Test Accuracy = 0.6438

Epoch 2: 	Train Loss = 0.6532, Train Accuracy = 0.6237
	 	 Test Loss = 0.6387,  Test Accuracy = 0.6539

Epoch 3: 	Train Loss = 0.6465, Train Accuracy = 0.6192
	 	 Test Loss = 0.6340,  Test Accuracy = 0.6616

Epoch 4: 	Train Loss = 0.6415, Train Accuracy = 0.6363
	 	 Test Loss = 0.6353,  Test Accuracy = 0.6514

Epoch 5: 	Train Loss = 0.6399, Train Accuracy = 0.6165
	 	 Test Loss = 0.6344,  Test Accuracy = 0.6565

Epoch 6: 	Train Loss = 0.6395, Train Accuracy = 0.6237
	 	 Test Loss = 0.6185,  Test Accuracy = 0.6641

Epoch 7: 	Train Loss = 0.6329, Train Accuracy = 0.6278
	 	 Test Loss = 0.6180,  Test Accuracy = 0.6463

Epoch 8: 	Train Loss = 0.6301, Train Accuracy = 0.6390
	 	 Test Loss = 0.6116,  Test Accuracy = 0.6336

Epoch 9: 	Train Loss = 0.6261, Train Accuracy = 0.6295
	 	 Test Loss = 0.6256,  Test Accuracy = 0.6692

Epoch 10: 	Train Loss = 0.6201, Train Accuracy = 0.6426
	 	 Test

In [14]:
folder = "custom_esm"
os.makedirs(folder, exist_ok=True)

# Create CSV filename inside the folder
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_log_path = os.path.join(folder, f"metrics_cesm_{timestamp}.csv")

# Now write the file
with open(csv_log_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
    writer.writeheader()
    writer.writerows(metrics_log)

print(f"✅ CSV saved to {csv_log_path}")

✅ CSV saved to custom_esm\metrics_cesm_20250501_015703.csv
